# COVID-19 newcomers, bellwethers and outbreak scientists (OpenAlex, May 2025)

## Overview
This notebook rebuilds every figure of the paper from the May 2025 OpenAlex snapshot.

* Preprocessing: works, authorships and concepts for 2015-2022, then the COVID-19 and outbreak-science subsets.
* Classification: outbreak scientists, bellwethers and newcomers, and the composition of each COVID-19 team.
* Figs 1-7 and the appendix venue tables, each written to a `.tex` file in `figures/`.
* The last section holds the queries added during revision.

Two choices affect the numbers. Journal two-year mean citedness (Fig 4) comes from the live OpenAlex API,
because the snapshot has no journal statistics, so it is a 2026 value, not a 2020-2022 one. Citation counts
(Fig 5) take citing-paper dates from the full snapshot, not from the 2015-2022 table, so citations made after
2022 are counted.

Contents: Setup; Preprocessing (needs the snapshot); Load the preprocessed tables; Fig 1; Author
classification; Fig 2; Fig 3; Citation counts (needs the snapshot); Fig 4 and the appendix venue tables;
Fig 5; Fig 6; Fig 7; Revision analyses. With the Zenodo deposit in `data/`, run Setup and the first code cell of
Preprocessing (it defines `work_types`), then skip the two sections marked as needing the snapshot.

## Setup
Paths come from environment variables; see the README.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import itertools
from notebook_utils import *
pd.options.future.infer_string = False
pio.templates.default = 'plotly_white+presentation'

In [ ]:
import os
repo_path = Path(os.environ.get('PANDEMIC_PUB_ROOT', '..')).resolve()  # repo root
pkl_path = repo_path / 'pickles'
parq_path = Path(os.environ['OPENALEX_SNAPSHOT'])  # May-2025 OpenAlex snapshot, parquet
csv_dir = Path(os.environ['OPENALEX_CSV'])         # May-2025 OpenAlex snapshot, csv entity tables
data_path = repo_path / 'data'
tex_path = repo_path / 'figures'  # .tex sources

In [ ]:
concept_ids_covid, concept_ids_outbreak_sc, start_date_tw, end_date_tw, start_date_ow, end_date_ow = get_topic_info('COVID')

## Preprocessing (needs the snapshot)
Keeps works and their authorships and concepts under these filters, and saves the three tables in `data/`.

* Publication year between the start of the TW and the end of the OW
* 1-30 authors; not retracted; not paratext
* Type in `{'article', 'journal-article', 'proceedings-article', 'posted-content', 'book-chapter', 'preprint', 'review', 'conference-paper'}`
* Authorships keep institution ids and assigned-institution rows
* Concepts keep rows with score above 0.30

The COVID-19 subset further requires 2 or more authors and a DOI; the outbreak-science subset does not.

### Works

In [ ]:
columns_d = dict(
  works=['work_id', 'doi', 'type', 'publication_date', 'publication_year', 'num_authors', 'language'],
  works_authorships=['work_id', 'author_id', 'institution_id'],
  works_concepts=['work_id', 'concept_id', 'score'],
)
work_types = {'article', 'journal-article', 'proceedings-article', 'posted-content', 'book-chapter', 'preprint', 'review', 'conference-paper'}

start_year, end_year = 2015, 2022

In [ ]:

works_filters = [[
    ('publication_year', '>=', start_year), 
    ('publication_year', '<=', end_year), 
    ('num_authors', '>=', 1),    # added Jul 2026
    ('num_authors', '<=', 30),   # added Jul 2026
    ('type', 'in', work_types),  # added Jul 2026
    ('is_retracted', '=', False), # added Jul 2026
    ('is_paratext', '=', False),  # added Jul 2026
]]
works_path = data_path / f'works_{start_year}_{end_year}.parquet'
works_df_2015_2022 = (
  pd.read_parquet(
    parq_path / 'works', 
    columns=columns_d['works'], engine='pyarrow', 
    filters=works_filters,
  )
)
works_df_2015_2022

In [ ]:
# Dates of 1 January are OpenAlex placeholders; spread them over the year, reproducibly (seed 0).
works_df_2015_2022 = shuffle_jan1_works(works_df_2015_2022, seed=0)
works_df_2015_2022.to_parquet(data_path / 'works_2015_2022.parquet')

Read the processed works table back.

### Authorships

In [ ]:
works_df_2015_2022 = read_parquet(data_path / 'works_2015_2022.parquet', columns=['work_id',  'publication_date', 'publication_year'])

In [ ]:
start_year, end_year = 2015, 2022
works_authors_2015_2022 = pd.read_parquet(
  parq_path / 'works_authorships',
  columns=['work_id', 'author_id', 'institution_id'],
  filters=[[
    ('publication_year', '>=', start_year), 
    ('publication_year', '<=', end_year), 
    ('assigned_institution', '=', True),
  ]],
)
works_authors_2015_2022

In [ ]:
work_ids = set(works_df_2015_2022.work_id)
print(f'Filtered work ids: {len(work_ids)=:,}')

display(f'Before {works_authors_2015_2022.shape[0]=:,}')
display(works_authors_2015_2022.head())
works_authors_2015_2022 = (
  works_authors_2015_2022
  .query('work_id.isin(@work_ids)')
  .merge(
      works_df_2015_2022[['work_id', 'publication_date', 'publication_year']],
      on='work_id', how='inner',
  )
)
display(f'After {works_authors_2015_2022.shape[0]=:,}')
display(works_authors_2015_2022.head())

works_authors_2015_2022.to_parquet(data_path / 'works_authors_2015_2022.parquet')

# Free before the concepts step. Both are dead once the parquet is written, but a
# 'Restart & Run All' keeps them resident (~10GB frame + ~5GB set) straight through the
# concepts merge, which is what exhausted memory and wedged the kernel on 2026-09-07.
del works_authors_2015_2022, work_ids
gc.collect()

### Concepts
Each preprocessing step reads its input from `data/` again, so a step can be rerun on its own.

In [ ]:
start_year, end_year = 2015, 2022
min_score = 0.30

works_concepts_2015_2022 = pd.read_parquet(
  parq_path / 'works_concepts',
  columns=['work_id', 'concept_id', 'score'],
  filters=[[
    ('publication_year', '>=', start_year), 
    ('publication_year', '<=', end_year), 
    ('score', '>', min_score),
  ]], 
)
display(works_concepts_2015_2022)

In [ ]:
works_df_2015_2022 = read_parquet(data_path / 'works_2015_2022.parquet', columns=['work_id',  'publication_date', 'publication_year'])

In [ ]:
display(f'Before {works_concepts_2015_2022.shape[0]=:,}')
display(works_concepts_2015_2022.head())

works_concepts_2015_2022 = (
  works_concepts_2015_2022
  .merge(
      works_df_2015_2022[['work_id', 'publication_year']],
      on='work_id', how='inner',
  )
)
display(f'After {works_concepts_2015_2022.shape[0]=:,}')
display(works_concepts_2015_2022.head())

works_concepts_2015_2022.to_parquet(data_path / 'works_concepts_2015_2022.parquet')

del works_concepts_2015_2022   # ~9GB, dead once written; cell 30 re-reads it filtered
gc.collect()

## Load the preprocessed tables

In [ ]:
works_df = read_parquet(data_path / 'works_2015_2022.parquet', columns=['work_id', 'doi', 'type', 'publication_year', 'publication_date', 'num_authors'])
works_authors_df = read_parquet(data_path / 'works_authors_2015_2022.parquet', columns=['work_id', 'author_id', 'publication_year'])
# Cells 33 and 35 are the only consumers of works_concepts_df and both immediately
# filter to these concepts. Filtering at read keeps ~1.9M rows (0.04GB) instead of
# 436M (8.7GB), and makes the downstream concept_id.isin() queries ~free (was ~40s).
works_concepts_df = read_parquet(data_path / 'works_concepts_2015_2022.parquet', columns=['work_id', 'concept_id', 'publication_year'],
                                 filters=[('concept_id', 'in', concept_ids_covid | concept_ids_outbreak_sc)])

### COVID-19 and outbreak-science subsets

_Specific to COVID papers, num_authors >= 2 and papers must have DOIs_

In [ ]:
covid_works_df = (
  works_df
  .query('num_authors>=2 & ~doi.isna()')   # THIS IS ONLY FOR COVID PAPERS
  .merge(
    works_concepts_df
    .query('concept_id.isin(@concept_ids_covid)')   
    .drop_duplicates(subset=['work_id'])            # drop duplicate work ids 
    [['work_id']], how='inner',
  )
)
display('COVID works sample', covid_works_df)
print(f'2015-2022 COVID works {covid_works_df.work_id.nunique():,}')

In [ ]:
covid_works_df.to_parquet(data_path / 'covid_works.parquet')

In [ ]:
## outbreak-science subset
outbreak_sci_works_df = (
  works_df
  .merge(
    works_concepts_df
    .query('concept_id.isin(@concept_ids_outbreak_sc)')
    .drop_duplicates(subset=['work_id'])    # drop duplicate work ids 
    [['work_id']], how='inner',
  )
)
display('Outbreak Sci works sample', outbreak_sci_works_df)
print(f'2015-2022 Outbreak Science works {outbreak_sci_works_df.work_id.nunique():,}')

In [ ]:
outbreak_sci_works_df.to_parquet(data_path / 'outbreak_science_works.parquet')

## Fig 1: yearly COVID-19 and outbreak-science output

In [ ]:
# NOTE: the .join() below is a LEFT join on publication_year, so the COVID leg drives
# the index. A year with outbreak-science works but zero COVID works would be dropped.
# Not triggered on 2015-2022 (thinnest COVID year is 2018 at ~900 works), but it would
# bite for a sparser topic. Duplicate authorship rows are safe here: nunique dedups.
yearly_counts_df = (
  works_authors_df
  .merge(
    covid_works_df
    [['work_id']],
    how='inner',
  )
  .groupby('publication_year')
  .agg(
    yearly_covid_works=('work_id', 'nunique'),
    yearly_covid_authors=('author_id', 'nunique'),
  )
  .join(
    works_authors_df
    .merge(
      outbreak_sci_works_df
      [['work_id']],
      how='inner',
    )
    .groupby('publication_year')
    .agg(
      yearly_outbreak_sci_works=('work_id', 'nunique'),
      yearly_outbreak_sci_authors=('author_id', 'nunique'),
    )
  )
)
yearly_counts_df

In [ ]:
# export to latex
print(
  yearly_counts_df
  .reset_index()
  .assign(
    topic='COVID',
  )
  .rename(
    columns={
      'publication_year': 'year',
      'yearly_covid_works': 'topic_works',
      'yearly_covid_authors': 'topic_authors',
      'yearly_outbreak_sci_works': 'exp_works',
      'yearly_outbreak_sci_authors': 'exp_authors',
    }
  )
  [['year', 'topic', 'topic_works', 'topic_authors', 'exp_works', 'exp_authors']]
  .to_csv(index=False, sep=' ')
)

## Author classification
Outbreak scientists, bellwethers and newcomers among the authors of COVID-19 papers in the OW (2020-2022).

In [ ]:
## authors with an outbreak-science or COVID-19 paper in the TW
outbreak_sci_tw_work_ids = set(
  outbreak_sci_works_df
  .query('publication_year<2020')  # before 2020
  .work_id
)

outbreak_sci_tw_author_ids = set(
  works_authors_df
  .query('work_id.isin(@outbreak_sci_tw_work_ids)')
  .author_id
)

print(f'Outbreak Science in TW: works {len(outbreak_sci_tw_work_ids):,}\t Authors: {len(outbreak_sci_tw_author_ids):,}')
## we need to add the COVID authors in the TW 
covid_tw_work_ids = set(
  covid_works_df
  .query('publication_year<2020')
  .work_id
)

covid_tw_author_ids = set(
  works_authors_df
  .query('work_id.isin(@covid_tw_work_ids)')
  .author_id
)
print(f'COVID in TW: works {len(covid_tw_work_ids):,}\t Authors: {len(covid_tw_author_ids):,}')

### Outbreak scientists
* Wrote a COVID-19 paper in the OW, and
* wrote an outbreak-science paper or a COVID-19 paper in the TW (2015-2019)

In [ ]:
# works authors info of COVID papers in the Observation Window
covid_ow_work_ids = set(
  covid_works_df
  .query('publication_year.between(2020, 2022)')
  .work_id
)

covid_ow_works_authors_df = (
  works_authors_df
  .query('publication_year.between(2020, 2022)')
  .query('work_id.isin(@covid_ow_work_ids)')
  .drop_duplicates(subset=['work_id', 'author_id'])  # drop institution info
)
print(f'COVID in OW (2020-2022): Works: {covid_ow_works_authors_df.work_id.nunique():,}\t Authors: {covid_ow_works_authors_df.author_id.nunique():,}')

covid_experts_ow_author_ids = set(covid_ow_works_authors_df.author_id) & (outbreak_sci_tw_author_ids | covid_tw_author_ids)
print(f'COVID expert authors: {len(covid_experts_ow_author_ids):,}')

### Bellwethers
* Wrote a COVID-19 paper in the OW, and
* wrote any paper in the TW, but none on outbreak science or COVID-19

In [ ]:
tw_author_ids = set(
  works_authors_df
  .query('publication_year<2020')
  .author_id
)
print(f'{len(tw_author_ids):,} Authors in TW (2015-19) with at least 1 paper')

# discard all authors who wrote a COVID / Outbreak Sci paper in TW 
covid_bellwethers_ow_author_ids = set(covid_ow_works_authors_df.author_id) & (tw_author_ids - outbreak_sci_tw_author_ids - covid_tw_author_ids)
print(f'{len(covid_bellwethers_ow_author_ids):,} Bellwethers (authors) in TW (2015-19) with at least 1 COVID paper in OW')

### Newcomers
* Wrote a COVID-19 paper in the OW, and
* wrote no paper in any field during the TW

In [ ]:
covid_newcomers_ow_author_ids = set(covid_ow_works_authors_df.author_id)  - covid_bellwethers_ow_author_ids - covid_experts_ow_author_ids
print(f'{len(covid_newcomers_ow_author_ids):,} Newcomers (authors): no paper in TW (2015-19), >=1 COVID paper in OW')

### COVID-19 OW tables
* COVID-19 papers in the OW only
* The works table gets `class_` (share of outbreak scientists) and `num_experts`, `num_bellwethers`, `num_newcomers`
* The authorships table gets each author's type and one boolean column per type

In [ ]:
condlist = [
  covid_ow_works_authors_df.author_id.isin(covid_experts_ow_author_ids),
  covid_ow_works_authors_df.author_id.isin(covid_bellwethers_ow_author_ids),
  covid_ow_works_authors_df.author_id.isin(covid_newcomers_ow_author_ids),
]
choicelist = [
  'expert',
  'bellwether',
  'newcomer'
]

covid_ow_works_authors_df = (
  covid_ow_works_authors_df
  .assign(
    author_type=lambda df_: np.select(condlist, choicelist=choicelist, default='ERROR'),
    is_expert=lambda df_: df_.author_id.isin(covid_experts_ow_author_ids),
    is_bellwether=lambda df_: df_.author_id.isin(covid_bellwethers_ow_author_ids),
    is_newcomer=lambda df_: df_.author_id.isin(covid_newcomers_ow_author_ids),
  )
)
covid_ow_works_authors_df

In [ ]:
covid_ow_works_authors_df.to_parquet(data_path / 'covid_works_authors_ow.parquet')

In [ ]:
## group each work into cats 
agg_covid_works_authors_df = (
  covid_ow_works_authors_df
  .groupby('work_id', sort=False, as_index=False)
  .agg(
    num_authors=('author_id', 'nunique'),
    num_experts=('is_expert', 'sum'),
    num_bellwethers=('is_bellwether', 'sum'),
    num_newcomers=('is_newcomer', 'sum'),
  )
  .assign(
    frac_expert=lambda df_: df_.num_experts.div(df_.num_authors),
    class_=lambda df_ : pd.cut(
                              df_.frac_expert,
                              bins=[0, 0.000001, 0.5, 0.999, 1],
                              include_lowest=True,
                              labels=['0%', '1-50%', '51-99%', '100%'],),
  )
)

agg_covid_works_authors_df

In [ ]:
covid_works_df = read_parquet(data_path / 'covid_works.parquet')

In [ ]:
## add to covid works table 
covid_ow_works_df = (
  covid_works_df
  .drop(columns=['num_experts', 'class_'], errors='ignore')
  .merge(
    agg_covid_works_authors_df
    [['work_id', 'num_experts', 'num_bellwethers', 'num_newcomers', 'class_']],
    on='work_id', how='inner',
  )
)
covid_ow_works_df

In [ ]:
covid_ow_works_df.to_parquet(data_path / 'covid_works_ow.parquet')

## Fig 2: authors and papers by group and year

In [ ]:
print('Panel A: Author splits')
print(
  covid_ow_works_authors_df
  .groupby(['publication_year', 'author_type'])
  .author_id
  .nunique()
  .rename('uniq_authors')
  .reset_index()
  .pivot(
    index='publication_year',
    columns='author_type',
    values='uniq_authors',
    # aggfunc='count',
  )
  .join(
    covid_ow_works_authors_df
    .groupby(['publication_year'])
    .author_id
    .nunique()
    .rename('total_authors')
  )
  .reset_index()
  .rename(columns={'publication_year': 'year'})
  .to_csv(sep=' ', index=False)
)

In [ ]:
(
  covid_ow_works_df
  .groupby(['publication_year', 'class_'], observed=True)
  .work_id
  .nunique()
  .reset_index()
  .pivot(
    index='publication_year', columns='class_',
    values='work_id',
  )
  .reset_index()
)

In [ ]:
print(
  covid_ow_works_df
  .groupby(['publication_year', 'class_'], observed=True)
  .work_id
  .nunique()
  .reset_index()
  .pivot(
    index='publication_year', columns='class_',
    values='work_id',
  )
  .reset_index()
  .merge(
    covid_works_df
    .reset_index()
    .groupby(['publication_year'])
    .work_id
    .nunique()
    .rename('total_works'),
    left_on='publication_year', right_index=True,
  )
  .rename(columns={'publication_year': 'year'})
  .to_csv(sep=' ', index=False)
  .replace('%', 'p')
)

## Fig 3: team size

In [ ]:
print('Team Size boxplot')
display(
  covid_ow_works_df
  .groupby('class_', observed=True)
  .num_authors
  .describe()
  .convert_dtypes()
  .round(3)
  .assign(
    IQR=lambda df_: df_.apply(lambda row: row['75%']-row['25%'], axis=1),
    lower_whisker=lambda df_: df_.apply(lambda row: max(row['min'], row['25%'] - 1.5*row['IQR']), axis=1),
    upper_whisker=lambda df_: df_.apply(lambda row: min(row['max'], row['75%'] + 1.5*row['IQR']), axis=1),
  )
  # .to_csv(sep=' ')
  # .replace('%', 'p')
)

## Citation counts (needs the snapshot)
Two-year citations of each COVID-19 paper in the OW. Citing works are any OpenAlex works of the eight types with 1-30 authors; their dates come from the full snapshot, so citations made after 2022 count.

In [ ]:
covid_ow_works_df = (
  pd.read_parquet(
    data_path / 'covid_works_ow.parquet',
    columns=['work_id', 'publication_date'],
  )
)
covid_ow_works_df

In [ ]:
## get list of works that cite a COVID paper
covid_ow_work_ids = set(covid_ow_works_df.work_id)

covid_works_refs_works_df = (
  pd.read_parquet(
    parq_path / 'works_referenced_works', 
    filters=[
      ['referenced_work_id', 'in', covid_ow_work_ids],  # only rows that cite a COVID paper in OW
    ],
  )
)
display(covid_works_refs_works_df)

covid_citing_work_ids = set(covid_works_refs_works_df.work_id)  # these papers cite a COVID paper
print(f'{len(covid_citing_work_ids):,} papers cite a COVID paper written in OW')

In [ ]:
## filter the citing work ids to the same constraints
## NOTE: read from the FULL snapshot, not works_2015_2022.parquet -- citing papers
## published after 2022 must be kept or num_cites_1yr/2yrs are right-censored.
## Matches jun25-preprint-figs.ipynb (the authoritative notebook).
sub_works_df = (
  pd.read_parquet(
    parq_path / 'works',
    columns=['work_id', 'publication_date'],
    filters=[
      ['work_id', 'in', covid_citing_work_ids],
      ['num_authors', '>', 0],
      ['num_authors', '<=', 30],
      ['type', 'in', work_types],
    ],
  )
)
sub_works_df

In [ ]:
# build a date dictionary -> COVID papers dates should come from shuffled date, citing papers should have dates from sub_works_df
date_dict = (
  sub_works_df
  .set_index('work_id')
  .publication_date
  .to_dict()
)

date_dict.update(
  covid_ow_works_df
  .set_index('work_id')
  .publication_date
  .to_dict()
)
print(f'Date dictionary {len(date_dict)=:,}')

In [ ]:
covid_works_refs_works_df = (
  covid_works_refs_works_df  # referenced_work_id MUST BE a COVID paper
  .query('work_id.isin(@date_dict)')
  .assign(
    # .map(dict) is ~5x faster than .progress_map(lambda) over ~8.6M rows (24s -> 4.5s)
    work_date=lambda df_: df_.work_id.map(date_dict),
    referenced_work_date=lambda df_: df_.referenced_work_id.map(date_dict),
  )
  .assign(
    time_diff_days=lambda df_: (df_.work_date - df_.referenced_work_date).dt.days,
    # Windowed counts keep the -366 backward slack to absorb shuffled Jan-1 dates.
    # Both bounds are explicit here on purpose: the old `.query('time_diff_days>=-366')`
    # supplied the lower bound for within_2_yrs, but it ALSO restricted num_cites_total.
    within_1_yr=lambda df_: df_.time_diff_days.between(-366, 366, inclusive='both'),
    within_2_yrs=lambda df_: df_.time_diff_days.between(-366, 732, inclusive='both'),
  )
  # NO row filter here -- num_cites_total must count every citation, unrestricted in time.
)
covid_works_refs_works_df

In [ ]:
covid_cite_stats_df = (
  covid_works_refs_works_df
  .groupby('referenced_work_id', sort=False)
  .agg(
    num_cites_total=('work_id', 'nunique'),   # ALL citations, no time window
    num_cites_1yr=('within_1_yr', 'sum'),     # lag in [-366, 366]
    num_cites_2yrs=('within_2_yrs', 'sum'),   # lag in [-366, 732]
  )
  .reset_index()
  .rename(columns={'referenced_work_id': 'work_id'})
)
covid_cite_stats_df

In [ ]:
covid_ow_works_df = read_parquet(data_path / 'covid_works_ow.parquet')

In [ ]:
covid_ow_works_df = (
  covid_ow_works_df
  .drop(columns=['num_cites_total', 'num_cites_1yr', 'num_cites_2yrs', 'num_cites_2yr'], errors='ignore')
  .merge(
    covid_cite_stats_df,
    on='work_id', how='left'
  )
  .assign(
    num_cites_total=lambda df_: df_.num_cites_total.fillna(0).astype('int'),
    num_cites_1yr=lambda df_: df_.num_cites_1yr.fillna(0).astype('int'),
    num_cites_2yrs=lambda df_: df_.num_cites_2yrs.fillna(0).astype('int'),
  )
)
display(covid_ow_works_df)

In [ ]:
covid_ow_works_df.to_parquet(data_path / 'covid_works_ow.parquet')

## Fig 4: Journal impact factor by team composition
* The venue of each COVID OW work is its primary location in `works_primary_location`.
* Only journals with an `issn_l` are kept; the deleted journal `4317411217` is dropped.
* Impact factor is `summary_stats.2yr_mean_citedness` from the OpenAlex API, cached in
  `data/covid_journal_sources.parquet` (delete it to refresh). The snapshot's `sources.csv.gz` has no
  summary statistics, and `sources_counts_by_year.csv.gz` cannot replace them: its yearly `cited_by_count`
  counts citations received in a year, not citations to the previous two years' works.
* The API does not return 977 of the 37,703 journals (merged or deleted ids), which drops 4,844 works (1.4%).
* Each work carries its journal's value, so a journal counts once per COVID-19 paper it published.

In [ ]:
covid_ow_works_df = (
  pd.read_parquet(
    data_path / 'covid_works_ow.parquet',
    columns=['work_id', 'publication_year', 'num_authors', 'class_'],
  )
)
covid_ow_work_ids = set(covid_ow_works_df.work_id)

covid_works_primary_location_df = (
  pd.read_parquet(
    parq_path / 'works_primary_location',
    columns=['work_id', 'source_id', 'source_name', 'source_type'],
    filters=[
      ['work_id', 'in', covid_ow_work_ids],
    ],
  )
)
print(f'{len(covid_ow_work_ids):,} COVID OW works, '
      f'{covid_works_primary_location_df.work_id.nunique():,} have a primary location')
covid_works_primary_location_df

In [ ]:
sources_df = (
  pd.read_csv(
    csv_dir / 'sources.csv.gz',
    usecols=['source_id', 'source_name', 'issn_l', 'publisher', 'works_count', 'cited_by_count', 'is_oa'],
  )
)
sources_df

In [ ]:
covid_venues_df = (
  covid_works_primary_location_df
  .query('source_id != 4317411217')       # deleted journal
  .merge(
    sources_df
    [['source_id', 'issn_l', 'publisher', 'works_count', 'cited_by_count', 'is_oa']],
    on='source_id',
  )
  .merge(
    covid_ow_works_df,
    on='work_id',
  )
  .query('~issn_l.isna()')                # sources without an ISSN-L
  .query('source_type=="journal"')
)
print(f'{covid_venues_df.work_id.nunique():,} works in '
      f'{covid_venues_df.source_id.nunique():,} journals')
covid_venues_df

In [ ]:
import requests

def parse_source_result(results):
  rows = []
  for res in results:
    summary_stats = res.get('summary_stats') or {}
    rows.append(
      {
        'source_id': convert_openalex_id_to_int(res['id']),
        'source_name': res['display_name'],
        'country_code': res['country_code'],
        'type': res['type'],
        'is_core': res.get('is_core'),
        'num_works': res['works_count'],
        'num_cites': res['cited_by_count'],
        'impact_factor': summary_stats.get('2yr_mean_citedness', pd.NA),
        'h_index': summary_stats.get('h_index', pd.NA),
        'i10_index': summary_stats.get('i10_index', pd.NA),
      }
    )
  return rows


def fetch_sources_info(source_ids, chunk_size=50):
  """Pull summary_stats for source_ids from the OpenAlex API, chunk_size ids per request.

  OpenAlex caps an OR-filter at 50 values, hence the default. Retries on 429/network errors.
  """
  rows = []
  for i in tqdm(range(0, len(source_ids), chunk_size)):
    chunk = source_ids[i: i + chunk_size]
    url = (
      'https://api.openalex.org/sources?filter=openalex:'
      + '|'.join(f'S{source_id}' for source_id in chunk)
      + '&per-page=200&mailto=' + os.environ.get('OPENALEX_MAILTO', '')
    )
    for attempt in range(5):
      try:
        response = requests.get(url, timeout=60)
        if response.status_code == 429:      # rate limited, back off
          time.sleep(5 * (attempt + 1))
          continue
        response.raise_for_status()
        break
      except Exception:
        if attempt == 4:
          raise
        time.sleep(3 * (attempt + 1))
    else:                                  # every attempt was rate limited
      raise RuntimeError(f'rate limited 5x on the chunk starting at {i}')
    rows.extend(parse_source_result(response.json()['results']))

  return pd.DataFrame(rows)

In [ ]:
## cached on disk -- delete the parquet to re-pull from the API
sources_info_path = data_path / 'covid_journal_sources.parquet'

if sources_info_path.exists():
  sources_info_df = pd.read_parquet(sources_info_path)
  print(f'Read {len(sources_info_df):,} journals from {sources_info_path.name!r}')
else:
  sources_info_df = fetch_sources_info(sorted(covid_venues_df.source_id.unique()))
  sources_info_df.to_parquet(sources_info_path)
  print(f'Wrote {len(sources_info_df):,} journals to {sources_info_path.name!r}')

print(f'{sources_info_df.impact_factor.isna().sum():,} journals have no 2yr_mean_citedness')
sources_info_df

In [ ]:
num_journal_works = covid_venues_df.work_id.nunique()   # before the API merge drops merged/deleted sources

covid_venues_df = (
  covid_venues_df
  .merge(
    sources_info_df
    [['source_id', 'country_code', 'num_cites', 'impact_factor', 'h_index', 'is_core']],
    on='source_id',
  )
)
print(f'{covid_venues_df.work_id.nunique():,} works in '
      f'{covid_venues_df.source_id.nunique():,} journals matched by the API; '
      f'{covid_venues_df.impact_factor.isna().sum():,} works sit in a journal with no impact factor')
covid_venues_df

In [ ]:
## where the works go missing, stage by stage
print(
  pd.Series(
    {
      'COVID OW works': len(covid_ow_works_df),
      'has a primary location': covid_works_primary_location_df.work_id.nunique(),
      'source is not the deleted journal': (
        covid_works_primary_location_df.query('source_id != 4317411217').work_id.nunique()
      ),
      'source has an ISSN-L': (
        covid_works_primary_location_df
        .merge(sources_df[['source_id', 'issn_l']], on='source_id')
        .query('~issn_l.isna()')
        .work_id.nunique()
      ),
      'source_type == journal': num_journal_works,
      'journal matched by the OpenAlex API': covid_venues_df.work_id.nunique(),
    },
    name='num_works',
  )
  .to_frame()
  .assign(
    frac=lambda df_: df_.num_works / len(covid_ow_works_df),
  )
  .to_string(formatters={'num_works': '{:,}'.format, 'frac': '{:.1%}'.format})
)

In [ ]:
print('Journal impact factor boxplot (one observation per work)')
display(
  covid_venues_df
  .groupby('class_', observed=True)
  .impact_factor
  .describe()
  .convert_dtypes()
  .round(3)
  .assign(
    IQR=lambda df_: df_.apply(lambda row: row['75%']-row['25%'], axis=1),
    lower_whisker=lambda df_: df_.apply(lambda row: max(row['min'], row['25%'] - 1.5*row['IQR']), axis=1),
    upper_whisker=lambda df_: df_.apply(lambda row: min(row['max'], row['75%'] + 1.5*row['IQR']), axis=1),
  )
)

In [ ]:
print(
  covid_venues_df
  .groupby('class_', observed=True)
  .impact_factor
  .describe()
  .round(3)
  .assign(
    IQR=lambda df_: df_.apply(lambda row: row['75%']-row['25%'], axis=1),
    lower_whisker=lambda df_: df_.apply(lambda row: max(row['min'], row['25%'] - 1.5*row['IQR']), axis=1),
    upper_whisker=lambda df_: df_.apply(lambda row: min(row['max'], row['75%'] + 1.5*row['IQR']), axis=1),
  )
  .reset_index()
  .astype({'class_': 'string', 'count': int})   # keep counts out of scientific notation
  .replace({'class_': {'0%': 0, '1-50%': 50, '51-99%': 99, '100%': 100}})   # numeric labels, as in Fig 5
  .to_csv(sep=' ', float_format='%.5g', index=False)
  .replace('%', 'p')
)

### Fig 4 sensitivity to venue quality
Four groups of venues could distort the figure:
* journals with impact factor 0 (4,855 journals, 15,386 works), 2.4 times more common in the 0% class than in
  51-99%; 46% of them published nothing after 2022
* venues outside OpenAlex's core set (`is_core` false), which hold 40% of 0%-class works against 8-16% for the
  other classes, with a mean impact factor around 0.4
* Cureus (`2738950867`) and Research, Society and Development (`2739064124`), with 1,340 and 1,067 works in the
  0% class
* journals with impact factor above 50 (13 journals, 26 works)

The ordering of the four classes holds under every cut. Restricting to core venues roughly halves the gap
between the 0% class and the others and removes the dip of the 100% class below 51-99%.

In [ ]:
bad_source_ids = [2738950867, 2739064124]   # Cureus (WoS paper-mill hold), Research Society and Development

display(
  pd.DataFrame(
    {
      label: (
        subset_df
        .groupby('class_', observed=True)
        .impact_factor
        .agg(['size', 'mean', 'median'])
        .round(2)
        .stack()
      )
      for label, subset_df in [
        ('all (as shipped)', covid_venues_df),
        ('impact_factor > 0', covid_venues_df.query('impact_factor > 0')),
        ('is_core only', covid_venues_df.query('is_core')),
        ('drop Cureus + RSD', covid_venues_df.query('~source_id.isin(@bad_source_ids)')),
      ]
    }
  )
  .unstack(1)
)

### Fig 4 against the preprint's numbers
The preprint's values were about 20% higher for all four classes. Swapping one input at a time shows why
(0% class):

| 0% class | mean | median |
|---|---|---|
| preprint works, Dec 2024 impact factors | 2.18 | 1.08 |
| these works, Dec 2024 impact factors | 2.13 | 1.03 |
| these works, 2026 impact factors (used) | 1.77 | 0.83 |

About 88% of the gap comes from the API's two-year window moving forward: 64% of journals' values fell
between the two pulls. The remainder is a smaller 100% class here (4,407 works against 5,346), which comes
from the author classification, not from venues. The cell below needs the preprint-era pull, which is not
included here.

In [ ]:
quartz_sources_info_path = repo_path / 'data' / 'preprint_sources_info.parquet'  # optional: preprint-era API pull, not deposited

if quartz_sources_info_path.exists():
  display(
    pd.DataFrame(
      {
        label: (
          covid_venues_df
          .drop(columns=['impact_factor'])
          .merge(impact_factor_df[['source_id', 'impact_factor']], on='source_id')
          .groupby('class_', observed=True)
          .impact_factor
          .agg(['size', 'mean', 'median'])
          .round(3)
          .stack()
        )
        for label, impact_factor_df in [
          ('Dec-2024 pull (preprint)', pd.read_parquet(quartz_sources_info_path)),
          ('2026 pull (shipped)', sources_info_df),
        ]
      }
    )
    .unstack(1)
  )
else:
  print(f'{quartz_sources_info_path} is gone -- skipping the preprint comparison')

### Appendix: top venues per work class
Writes the data for two sets of appendix tables, from Fig 4's venue population less the 11 journals with no
2020-2022 row in `sources_counts_by_year` (18 works):
* `top20-venues-agg.txt`: top 20 journals per class over 2020-2022 (4 classes, 20 rows each)
* `top10-venues-yearly.txt`: top 10 per class per year (4 classes, 3 years, 10 rows each)

Both files are semicolon-separated without quoting. The LaTeX tables read class blocks by row position, so the
block order (`0p, 1-50p, 51-99p, 100p`) and the row counts are fixed; the asserts below check both. Journals are
ranked by COVID-19 work count, ties share the lower rank, and `source_name` breaks ties. `works_count`,
`works_count_2020_2022` and `works_count_year` are OpenAlex's own counts for the journal, not counts from our
sample.

In [ ]:
import re

## Appendix venue tables: journal output per year, straight from OpenAlex's own counts.
## Not our filtered works table: these columns describe the journal, not our sample.
counts_by_year_df = (
  pd.read_csv(csv_dir / 'sources_counts_by_year.csv.gz', usecols=['source_id', 'year', 'works_count'])
  .query('2020 <= year <= 2022')
  .rename(columns={'works_count': 'works_count_year'})
)

## journal names are long; p{6.5cm} at \scriptsize fits ~45 characters before wrapping, so only
## names past that get abbreviated (otherwise 'Science' becomes a bare 'Sci.'). Word-level,
## longest form first (Sciences before Science), \b-anchored so a word inside a word is left alone.
name_abbreviations = {
  'International': 'Int.',
  'Environmental': 'Env.',
  'Research': 'Res.',
  'Public': 'Pub.',
  'Sciences': 'Sci.',
  'Science': 'Sci.',
  'Proceedings': 'Proc.',
  'University': 'Univ.',
  'Association': 'Assoc.',
}
abbreviate_pattern = re.compile(r'\b(' + '|'.join(name_abbreviations) + r')\b')
max_name_length = 40
abbreviate = lambda s: (
  abbreviate_pattern.sub(lambda m: name_abbreviations[m.group()], s) if len(s) > max_name_length else s
)

venue_stats_df = (
  covid_venues_df
  .drop(columns=['source_name'])                  # per-work name from works_primary_location
  .merge(                                         # canonical snapshot name, one per source_id
    sources_df[['source_id', 'source_name']].merge(
      sources_info_df[['source_id', 'i10_index']], on='source_id',
    ),
    on='source_id',
  )
  .merge(
    counts_by_year_df.groupby('source_id').works_count_year.sum().rename('works_count_2020_2022'),
    on='source_id',
  )
  .assign(
    source_type=lambda df_: df_.source_type.str.title(),
    class_=lambda df_: df_.class_.astype(str).str.replace('%', 'p', regex=False),
  )
)
print(f'{venue_stats_df.work_id.nunique():,} works in {venue_stats_df.source_id.nunique():,} journals')

class_order = ['0p', '1-50p', '51-99p', '100p']
journal_cols = ['source_id', 'source_name', 'publisher', 'source_type', 'works_count',
                'works_count_2020_2022', 'num_cites', 'country_code', 'impact_factor',
                'h_index', 'i10_index']


def top_journals(group_cols, n):
  """Top-n journals by COVID work count within each group; ties share the lower rank."""
  return (
    venue_stats_df
    .groupby(group_cols + journal_cols, observed=True)
    .work_id.nunique().rename('num_covid_works').reset_index()
    ## source_name is the tie-break, so the cut at rank n is deterministic run to run
    .sort_values(group_cols + ['num_covid_works', 'source_name'],
                 ascending=[True] * len(group_cols) + [False, True])
    .assign(rank=lambda df_: df_.groupby(group_cols).num_covid_works
                                .rank(method='min', ascending=False).astype(int))
    .groupby(group_cols, observed=True).head(n)
    .assign(
      class_=lambda df_: pd.Categorical(df_.class_, categories=class_order, ordered=True),
      source_name=lambda df_: df_.source_name.map(abbreviate),
      publisher=lambda df_: df_.publisher.fillna('').map(abbreviate),
    )
    .sort_values(group_cols + ['rank'])
  )


agg_venues_df = top_journals(['class_'], 20)[
  ['class_', 'rank', 'source_id', 'source_name', 'publisher', 'source_type', 'num_covid_works',
   'works_count', 'works_count_2020_2022', 'num_cites', 'country_code', 'impact_factor',
   'h_index', 'i10_index']
]

## the yearly tables report the journal's output in that year, so works_count_year joins on (source, year)
yearly_venues_df = (
  top_journals(['class_', 'publication_year'], 10)
  .rename(columns={'publication_year': 'year'})
  .merge(counts_by_year_df, on=['source_id', 'year'])
  [['class_', 'year', 'rank', 'source_id', 'source_name', 'publisher', 'source_type',
    'num_covid_works', 'works_count', 'works_count_year', 'works_count_2020_2022', 'num_cites',
    'country_code', 'impact_factor', 'h_index', 'i10_index']]
)
display(agg_venues_df.head())
display(yearly_venues_df.head())

In [ ]:
## Write the two pgfplotstable data files read by tex/tables-holder.tex.
## Class blocks are written in 0p / 1-50p / 51-99p / 100p order and the holder slices them by row
## index (`skip rows between index`), so the row counts below are load-bearing: 4 classes x 20 rows
## for the aggregate tables, 4 classes x 3 years x 10 rows for the yearly ones.
for df_ in (agg_venues_df, yearly_venues_df):
  for col in df_.select_dtypes('object'):
    bad = df_[col].astype(str).str.contains(';', regex=False)
    assert not bad.any(), f'{col!r} holds a semicolon, which would shift every later column: {df_.loc[bad, col].tolist()}'
  bad = df_.source_name.str.contains(r'[&%_#$\\{}~^]', regex=True)
  assert not bad.any(), f'LaTeX specials in source_name: {df_.loc[bad, "source_name"].tolist()}'
  assert df_.notna().all().all(), f'nulls in {df_.columns[df_.isna().any()].tolist()}'

assert len(agg_venues_df) == 80, len(agg_venues_df)
assert len(yearly_venues_df) == 120, len(yearly_venues_df)

agg_venues_df.to_csv(tex_path / 'top20-venues-agg.txt', sep=';', index=False)
yearly_venues_df.to_csv(tex_path / 'top10-venues-yearly.txt', sep=';', index=False)
print(f'Wrote top20-venues-agg.txt ({len(agg_venues_df)} rows) and '
      f'top10-venues-yearly.txt ({len(yearly_venues_df)} rows) to {tex_path}')

## Fig 5: citations

In [ ]:
covid_ow_works_df = read_parquet(data_path / 'covid_works_ow.parquet')  # with citation counts

In [ ]:
(
  covid_ow_works_df
  .groupby('class_', observed=False)
  .agg(
    num_works=('work_id', 'count'),
    avg_cites_total=('num_cites_total', 'mean'),
    avg_cites_1yr=('num_cites_1yr', 'mean'),
    avg_cites_2yrs=('num_cites_2yrs', 'mean'),
    zero_cites_1yr=('num_cites_1yr', lambda ser: (ser==0).sum()),
    zero_cites_2yrs=('num_cites_2yrs', lambda ser: (ser==0).sum()),
  )
)

In [ ]:
(
  covid_ow_works_df
  .groupby(['class_', 'publication_year'], observed=False)
  .agg(
    num_works=('work_id', 'count'),
    avg_cites_total=('num_cites_total', 'mean'),
    avg_cites_1yr=('num_cites_1yr', 'mean'),
    avg_cites_2yrs=('num_cites_2yrs', 'mean'),
    zero_cites_1yr=('num_cites_1yr', lambda ser: (ser==0).sum()),
    zero_cites_2yrs=('num_cites_2yrs', lambda ser: (ser==0).sum()),
  )
)

In [ ]:
cites_by_group = (
  covid_ow_works_df
  .reset_index()
  .groupby(['class_', 'publication_year'], observed=False, as_index=False)
  .agg(
    num_works=('work_id', 'count'),
    
    avg_cites_total=('num_cites_total', 'mean'),
    avg_cites_1yr=('num_cites_1yr', 'mean'),
    avg_cites_2yrs=('num_cites_2yrs', 'mean'),

    zero_cites_total=('num_cites_total', lambda ser: (ser==0).sum()),
    zero_cites_total_=('num_cites_total', lambda ser: (ser==0).sum() / ser.size),
    zero_cites_1yr=('num_cites_1yr', lambda ser: (ser==0).sum()),
    zero_cites_1yr_=('num_cites_1yr', lambda ser: (ser==0).sum() / ser.size),
    zero_cites_2yrs=('num_cites_2yrs', lambda ser: (ser==0).sum()),
    zero_cites_2yrs_=('num_cites_2yrs', lambda ser: (ser==0).sum() / ser.size),
  )
)

cites_by_group

In [ ]:
## conf_interval
cite_rows = []

for year, year_grp in tqdm(covid_ow_works_df.groupby('publication_year', observed=True)):
  for typ, work_typ_grp in tqdm(year_grp.groupby('class_', observed=True)):
    # display(year, typ, work_typ_grp.head(2))
    counts = work_typ_grp.num_cites_2yrs.fillna(0).astype(int)
    num_cites_stats = conf_interval(counts, return_errors=True).to_dict()

    zero_cites_num = (counts==0).sum()
    
    row = {
      'year': year, 'work_type': typ, 'num_works': work_typ_grp.work_id.nunique(),   # by work_id, not index identity
      'num_uncited_2yrs': zero_cites_num,
      'num_cites_2yrs_median': counts.median(),
      'num_cites_2yrs_mean': num_cites_stats['mean'], 
      'num_cites_2yrs_ci95_min': num_cites_stats['mean_ci95_error_min'], 
      'num_cites_2yrs_ci95_max': num_cites_stats['mean_ci95_error_max'], 
    }

    cite_rows.append(row)
    # break
  # break

cite_stats_df = pd.DataFrame(cite_rows)
cite_stats_df

In [ ]:
print(
  cite_stats_df
  .replace(
    {
      'work_type': {'0%': 0, '1-50%': 50, '51-99%': 99, '100%': 100},
    },
  )
  .to_csv(sep=' ', float_format='%.5g', index=False)
  .replace('%', 'p')
)

## Fig 6: UpSet plot of outbreak scientists' prior topics
For each outbreak scientist, which of the four prior-experience concepts they published under during the TW
(2015-2019): Infectious disease (`524204448`), Epidemiology (`107130276`), Outbreak (`116675565`) and
Pandemic (`89623803`).

The intersection bars are exclusive: each author is counted once, in the bar for exactly the combination of
concepts they published under. The set-size bars give each concept's full membership, including authors shared
with other concepts, so they sum to more than the number of authors. They are passed to the plot directly rather
than summed from the intersection rows, which would be wrong for overlapping sets.

An earlier version of this figure (preprint) plotted union sizes as if they were intersections. The two cells
after this section rebuild the union view for comparison.

In [ ]:
## which of the four prior-experience concepts each author published under, in the TW
upset_buckets = {524204448: 'ID', 107130276: 'Epi', 116675565: 'Outbreak', 89623803: 'Pandemic'}

outbreak_sci_works_df = pd.read_parquet(data_path / 'outbreak_science_works.parquet',
                                        columns=['work_id', 'publication_year'])

# works_concepts_2015_2022 already carries the score>0.30 cut and is inner-joined to works_df
tw_concepts_df = (
  pd.read_parquet(
    data_path / 'works_concepts_2015_2022.parquet',
    columns=['work_id', 'concept_id', 'publication_year'],
    filters=[('concept_id', 'in', set(upset_buckets))],
  )
  .query('publication_year < 2020')
)

tw_authors_df = (
  pd.read_parquet(
    data_path / 'works_authors_2015_2022.parquet',
    columns=['work_id', 'author_id', 'publication_year'],
    filters=[('publication_year', '<', 2020)],
  )
)

author_bucket_df = (
  tw_concepts_df
  [['work_id', 'concept_id']]
  .merge(tw_authors_df[['work_id', 'author_id']], on='work_id')
  [['author_id', 'concept_id']]
  .drop_duplicates()
)
print(f'{len(author_bucket_df):,} author-bucket pairs over '
      f'{author_bucket_df.author_id.nunique():,} authors')

In [ ]:
expert_author_ids = set(
  pd.read_parquet(data_path / 'covid_works_authors_ow.parquet', columns=['author_id', 'is_expert'])
  .query('is_expert')
  .author_id
)

upset_df = (
  author_bucket_df
  .query('author_id.isin(@expert_author_ids)')
  .assign(bucket=lambda df_: df_.concept_id.map(upset_buckets), present=True)
  .pivot_table(index='author_id', columns='bucket', values='present', aggfunc='any')
  .reindex(columns=['Pandemic', 'ID', 'Outbreak', 'Epi'])
  .fillna(False)
  .astype(bool)
)
print(f'{len(expert_author_ids):,} COVID experts, {len(upset_df):,} of them published under at least '
      f'one of the four concepts in the TW')
print(f'{len(expert_author_ids) - len(upset_df):,} experts qualify only via a COVID paper in the TW, '
      f'so they have no bucket and are left out of the plot')
print('\nSet sizes (an author counted once per set they appear in):')
upset_set_sizes = upset_df.sum().sort_values(ascending=False)
print(upset_set_sizes.to_string())

## the Set size panel must be the true cardinality of each set. For DISJOINT cells that equals
## the sum of every cell containing the set -- assert it rather than trusting the plotting code.
## (It is emphatically NOT true for the union series, which is why the old figure's panel was wrong.)
_cells = upset_df.groupby(list(upset_df.columns)).size()
for _set in upset_df.columns:
  _from_cells = sum(n for idx, n in _cells.items() if dict(zip(_cells.index.names, idx))[_set])
  assert _from_cells == upset_set_sizes[_set], (_set, _from_cells, upset_set_sizes[_set])
print(f'\nSet sizes check out; they sum to {upset_set_sizes.sum():,} over a population of '
      f'{len(upset_df):,} because an author is counted once per set they belong to.')

In [ ]:
upset_counts = (
  upset_df
  .groupby(list(upset_df.columns))
  .size()
  .rename('counts')
  .sort_values(ascending=False)
)

display(
  upset_counts
  .to_frame()
  .assign(pct=lambda df_: (df_.counts / df_.counts.sum()).mul(100).round(2))
)
print(f'total {upset_counts.sum():,}, each expert counted once')

In [ ]:
fig = plot_upset(upset_counts, upset_set_sizes, ax_title='# Outbreak Scientists')
fig.savefig(repo_path / 'upset-plot-new.pdf', bbox_inches='tight')
fig.savefig(repo_path / 'upset-plot-new-preview.png', bbox_inches='tight', dpi=140)

### Comparison with the earlier figure (unions)
Summing the exclusive cells back into unions reproduces the earlier figure: the union of all four concepts is
100,991 here against 100,736 there, and every cell involving Epidemiology is within about 4%. The narrow
concepts moved more between snapshots (Infectious disease -41% and Pandemic -31% as single concepts), so
single-concept numbers from the preprint should not be compared directly.

In [ ]:
## unions rebuilt from the disjoint cells, next to the old figure's values
old_union_counts = {   # csvs/COVID_experts_breakdown.csv, as plotted in upset-plot-old.pdf
  (): 0, ('Outbreak',): 34727, ('Epi',): 82850, ('Epi', 'Outbreak'): 96381, ('Pandemic',): 13869,
  ('Pandemic', 'Outbreak'): 37222, ('Pandemic', 'Epi'): 85430, ('Pandemic', 'Epi', 'Outbreak'): 98029,
  ('ID',): 16797, ('ID', 'Outbreak'): 38955, ('ID', 'Epi'): 87069, ('ID', 'Epi', 'Outbreak'): 99208,
  ('ID', 'Pandemic'): 20557, ('ID', 'Pandemic', 'Outbreak'): 41243, ('ID', 'Pandemic', 'Epi'): 89291,
  ('ID', 'Pandemic', 'Epi', 'Outbreak'): 100736,
}

def union_size(bucket_names):
  """How many authors appear in at least one of bucket_names, summed from the disjoint cells."""
  return sum(
    n for idx, n in upset_counts.items()
    if any(flag for name, flag in zip(upset_counts.index.names, idx) if name in set(bucket_names))
  )

display(
  pd.DataFrame(
    [
      {
        'combination': ' | '.join(bucket_names) or '(none)',
        'n_sets': len(bucket_names),
        'old_union': old_value,
        'new_union': union_size(bucket_names),
      }
      for bucket_names, old_value in old_union_counts.items()
    ]
  )
  .assign(   # the (none) row has no baseline, so its delta is NA
    delta_pct=lambda df_: (df_.new_union - df_.old_union).div(df_.old_union.where(df_.old_union > 0)).mul(100).round(1),
  )
  .sort_values(['n_sets', 'old_union'])
  .set_index('combination')
)

### Union version of the plot
The same authors and concepts, with each bar counting authors who published under at least one of the marked
concepts. The set-size bars show real set sizes (79,577, 29,312, 9,920 and 9,601). A union chart always grows as
concepts are added, so it says little about composition; the exclusive version above is the one in the paper.

In [ ]:
upset_union_counts = (
  pd.Series(
    {
      flags: sum(
        n for idx, n in upset_counts.items()
        if any(f for name, f in zip(upset_counts.index.names, idx)
               if name in {nm for nm, sel in zip(upset_counts.index.names, flags) if sel})
      )
      for flags in itertools.product([False, True], repeat=upset_counts.index.nlevels)
      if any(flags)          # the empty selection has no union
    },
    name='counts',
  )
  .rename_axis(upset_counts.index.names)
  .sort_values(ascending=False)
)

fig = plot_upset(
  upset_union_counts, upset_set_sizes,
  ax_title='# Outbreak Scientists\nin $\\geq$1 marked set',
)
fig.savefig(repo_path / 'upset-plot-union.pdf', bbox_inches='tight')
fig.savefig(repo_path / 'upset-plot-union-preview.png', bbox_inches='tight', dpi=140)

### Fig 6 to LaTeX
Writes a self-contained pgfplots figure for each version, with the data inlined. It needs
`\usepackage{pgfplots}` and `\pgfplotsset{compat=1.18}`; the colors are defined inside the figure. Tick labels
use `scaled y ticks={real:1000}` because `\pgfmathparse` overflows above 16384.

In [ ]:
## Fig 6 for the paper: disjoint cells, bars sorted by cardinality (descending).
## The union variant stays available via upset_union_counts if a supplement wants it.

def cell_size(*bucket_names):
  """Size of the exclusive cell for exactly these buckets -- used to keep the caption in sync."""
  wanted = set(bucket_names)
  return next(
    n for idx, n in upset_counts.items()
    if {name for name, flag in zip(upset_counts.index.names, idx) if flag} == wanted
  )

## Names to print for each bucket. Only the wording is editorial -- the concept levels below come
## from the snapshot's concepts table (dict_concept_id_level, read by notebook_utils at import),
## so a snapshot that reshuffles the hierarchy updates the caption instead of contradicting it.
upset_bucket_labels = {
  'Epi': 'Epidemiology',
  'Outbreak': 'Outbreak',
  'ID': 'Infectious disease',
  'Pandemic': 'Pandemic',
}
concept_id_of_bucket = {short: concept_id for concept_id, short in upset_buckets.items()}

levels_clause = ', '.join(
  f'{upset_bucket_labels[short]} at level~{dict_concept_id_level[concept_id_of_bucket[short]]}'
  for short in sorted(
    upset_bucket_labels,
    key=lambda s: (dict_concept_id_level[concept_id_of_bucket[s]], upset_bucket_labels[s]),
  )
)

caption = (
  'Prior experience of COVID-19 outbreak scientists, by the concepts they published under during '
  '2015--2019. Bars are \\emph{exclusive} intersections: each author is counted once, in exactly '
  'the combination marked, so the bars partition all '
  f'{upset_counts.sum():,} authors. A combination therefore \\emph{{excludes}} the unmarked '
  f'concepts, and bars need not shrink as concepts are added --- {cell_size("Epi", "ID", "Pandemic"):,} '
  'authors published on Epidemiology, Infectious disease and Pandemic but \\emph{not} Outbreak, '
  f'while {cell_size("Epi", "ID", "Outbreak", "Pandemic"):,} published on all four. The horizontal '
  'bars at left give each concept\'s total size, which counts an author once per concept they '
  'appear in and so sums to more than the population. The four concepts sit at different levels of '
  f'the OpenAlex hierarchy ({levels_clause}), so the dominance of Epidemiology is partly a '
  'granularity effect.'
)

latex = upset_to_latex(
  upset_counts, upset_set_sizes,
  ylabel='\\# Outbreak Scientists',
  caption=caption,
  label='fig:upset-disjoint',
)
(tex_path / 'fig-upset-disjoint.tex').write_text(latex)
print(f'wrote fig-upset-disjoint.tex ({len(latex.splitlines()):,} lines)')
print()
print(caption)

## Fig 7: Collaboration mixing among the three author groups
A 3x3 mixing matrix over co-authorship edges among COVID-19 papers in the OW.

* An edge is one pair of co-authors on one paper, so a pair with three shared papers counts three times. A
  paper with $n_i$ authors of type $i$ adds $\binom{n_i}{2}$ edges within type $i$ and $n_i n_j$ edges between
  types $i$ and $j$.
* A: observed share of edges in each cell.
* B: expected share when author types are permuted across authorship slots, which keeps every team size and the
  overall type mix. The expected share is $q_i^2$ on the diagonal and $2 q_i q_j$ off it, with $q_i$ the share of
  authorship slots held by type $i$. A permutation test below checks this formula.
* C: A minus B, in percentage points.

$q$ is measured on authorship slots, not on unique authors: outbreak scientists are 7.7% of authors but hold
16.6% of slots because they publish more papers. Counts come from `covid_works_authors_ow`, because 1,150 works
have more authors in `num_authors` than classified authors.

In [ ]:
covid_works_authors_ow = read_parquet(data_path / 'covid_works_authors_ow.parquet')

In [ ]:
## Fig 7 mixing matrix: observed, analytic null, difference.
author_types = ['expert', 'bellwether', 'newcomer']

## authors of each type per work, one row per work
work_type_counts = (
  covid_works_authors_ow
  .assign(v=1)
  .pivot_table(index='work_id', columns='author_type', values='v', aggfunc='sum', fill_value=0)
  [author_types]
  .astype('int64')
)

n = work_type_counts.values
mixing_counts = n.T @ n                                     # off-diagonal: sum_w n_i n_j
np.fill_diagonal(mixing_counts, (np.diag(mixing_counts) - n.sum(0)) // 2)   # diagonal: sum_w C(n_i, 2)
num_edges = np.triu(mixing_counts).sum()

q = n.sum(0) / n.sum()                                      # authorship-slot share per type
null_counts = num_edges * np.outer(q, q) * (2 - np.eye(3))  # q_i^2 on the diagonal, 2 q_i q_j off it

collab_observed = pd.DataFrame(mixing_counts, index=author_types, columns=author_types)
collab_null     = pd.DataFrame(null_counts,   index=author_types, columns=author_types)
collab_diff     = (collab_observed - collab_null) / num_edges

print(f'{num_edges = :,}  (co-authorship edges, multi-edge)')
print('authorship-slot shares q:', dict(zip(author_types, q.round(4))))
display(
  (collab_observed / num_edges * 100).round(2).style.set_caption('A. Observed %'),
  (collab_null / num_edges * 100).round(2).style.set_caption('B. Shuffled %'),
  (collab_diff * 100).round(2).style.set_caption('C. Difference (pp)'),
)

In [ ]:
## Check on the analytic null: shuffle author_type across authorship slots for real and remeasure.
## Agrees with collab_null to ~0.03pp, which is the Monte-Carlo noise at 3 trials.
rng = np.random.default_rng(0)
work_idx = pd.factorize(covid_works_authors_ow.work_id.values)[0]
labels = covid_works_authors_ow.author_type.values

mc = np.zeros((3, 3))
trials = 3
for _ in range(trials):
  shuffled = (
    pd.DataFrame({'w': work_idx, 'author_type': rng.permutation(labels), 'v': 1})
    .pivot_table(index='w', columns='author_type', values='v', aggfunc='sum', fill_value=0)
    [author_types]
    .values
  )
  m = shuffled.T @ shuffled
  np.fill_diagonal(m, (np.diag(m) - shuffled.sum(0)) // 2)
  mc += m
mc /= trials

max_gap = np.abs(mc - null_counts).max() / num_edges * 100
print(f'largest analytic-vs-Monte-Carlo gap: {max_gap:.3f} pp')
assert max_gap < 0.1, 'analytic null disagrees with the permutation null'
display((pd.DataFrame(mc, index=author_types, columns=author_types) / num_edges * 100).round(2)
        .style.set_caption('B. Shuffled %, Monte Carlo'))

### Fig 7 as a matplotlib figure
A quick check of the numbers before they go to LaTeX. Only the lower triangle is drawn, since the matrix is
symmetric. Panels A and B share a 0-40% scale; C uses a diverging scale centered on zero.

In [ ]:
## Fig 7: 3-panel mixing heatmap. Lower triangle only (the matrix is symmetric).
collab_tick_labels = ['Outbr. Sc.', 'Bellwether', 'Newcomer']
mask_upper = np.triu(np.ones((3, 3), dtype=bool), k=1)

panels = [
  ('A. Original',  collab_observed / num_edges * 100, 'viridis', (0, 40),  '{:.1f}%'),
  ('B. Shuffled',  collab_null     / num_edges * 100, 'viridis', (0, 40),  '{:.1f}%'),
  ('C. Difference', collab_diff * 100,                'coolwarm', None,    '{:+.1f}'),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), constrained_layout=True)
for ax, (title, frame, cmap, clim, fmt) in zip(axes, panels):
  values = np.ma.masked_array(frame.values, mask_upper)
  if clim is None:
    bound = np.abs(values).max()
    clim = (-bound, bound)
  im = ax.imshow(values, cmap=cmap, vmin=clim[0], vmax=clim[1])
  ax.set_title(title, fontweight='bold', pad=12)
  ax.set_xticks(range(3), collab_tick_labels)
  ax.set_yticks(range(3), collab_tick_labels)
  ax.xaxis.set_ticks_position('top')
  ax.tick_params(length=0)
  ax.set_xticks(np.arange(-.5, 3), minor=True)
  ax.set_yticks(np.arange(-.5, 3), minor=True)
  ax.grid(which='minor', color='white', linewidth=2)
  for spine in ax.spines.values():
    spine.set_visible(False)

  ## white text on the dark end of the scale, black on the light end
  norm = im.norm
  for (row, col), value in np.ndenumerate(frame.values):
    if mask_upper[row, col]:
      continue
    dark = norm(value) < 0.35 if cmap == 'viridis' else not 0.25 < norm(value) < 0.75
    ax.text(col, row, fmt.format(value), ha='center', va='center',
            color='white' if dark else 'black', fontsize=11)

  fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
               format='%.0f%%' if cmap == 'viridis' else '%+.0f')

fig.suptitle('Collaboration mixing among the three author subgroups', fontsize=14)
plt.show()

### Fig 7 to LaTeX
Writes `collab-heatmap.tex` by replacing the three data blocks of the earlier figure's source, so the layout
and colormaps are unchanged. Panel C's cells and colorbar share one symmetric range. The figure needs:

```latex
\usepackage{pgfplots}
\usepackage{pgfplotstable}
\pgfplotsset{compat=1.18}
\usepgfplotslibrary{colormaps,colorbrewer,groupplots}
\usepackage{opencolor}
```

In [ ]:
## Fig 7 to LaTeX: swap the data blocks of the old figure, keep its styling.
import re

def pgfplots_table(counts, total, kind, fmt):
  ## old file's layout: x >= y filled, upper triangle nan; 0 outbreak sci, 1 bellwether, 2 newcomer
  rows = ['x y count kind sum']
  for y in range(3):
    for x in range(3):
      value = fmt(counts[x, y]) if x >= y else 'nan'
      rows.append(f'{x} {y} {value} {kind} {total}')
  return '\n'.join(rows)

blocks = {
  'origCounts':      pgfplots_table(collab_observed.values, num_edges, 'orig',     lambda v: f'{v:.0f}'),
  'shuffledCounts':  pgfplots_table(collab_null.values,     num_edges, 'shuffled', lambda v: f'{v:.2f}'),
  'diffNormCounts':  pgfplots_table(collab_diff.values,     0,         'shuffled', lambda v: f'{v:.4f}'),
}

tex = (tex_path / 'old-collab-heatmap.tex').read_text()
for name, block in blocks.items():
  tex, n_subs = re.subn(
    rf'(?m)^\\pgfplotstableread\{{[^{{}}]*\}}\{{\\{name}\}}',  # ^ skips the commented-out copy at the top
    lambda _m, b=block, nm=name: '\\pgfplotstableread{\n' + b + '\n}{\\' + nm + '}',
    tex,
  )
  assert n_subs == 1, f'{name}: expected 1 table block, replaced {n_subs}'

## the paper's axis colour: opencolor's oc-blue-8 in place of the old NDblue
tex = tex.replace('NDblue', 'oc-blue-8')

## Panel A prints its cells with `precision=1` but no `fixed zerofill`, so 20.04 comes out as "20%"
## next to "34.7%". Panel B already zerofills; match it.
tex, n_subs = re.subn(
  r'(title=\{A\. Original\}.*?nodes near coords style=\{font=\\small\\sffamily, /pgf/number format/\.cd,fixed,)precision=1',
  r'\1fixed zerofill,precision=1',
  tex, flags=re.S)
assert n_subs == 1, f'panel A number format: replaced {n_subs}'

## Panel C's colour scale. The old file ran the cells on -5..8 while its colourbar was labelled
## -10..10, so a cell's colour did not mean what the bar said it meant. Put both on the same
## symmetric range instead, wide enough for the largest deviation.
diff_pp = collab_diff.values * 100
diff_bound = int(np.ceil(np.abs(diff_pp).max()))
tex, n_subs = re.subn(r'point meta min=-5,\n        point meta max=8,',
                      f'point meta min=-{diff_bound},\n        point meta max={diff_bound},', tex)
assert n_subs == 1, f'panel C meta range: replaced {n_subs}'

tex, n_subs = re.subn(r'point meta min=-10, point meta max=10,',
                      f'point meta min=-{diff_bound}, point meta max={diff_bound},', tex)
assert n_subs == 1, f'panel C colourbar range: replaced {n_subs}'

diff_ticks = np.linspace(-diff_bound, diff_bound, 5)
tex, n_subs = re.subn(
  r'ytick=\{-10,-5,0,5,10\},\n            yticklabels=\{-10\\%,-5\\%,0\\%,5\\%,10\\%\},',
  'ytick={' + ','.join(f'{t:g}' for t in diff_ticks) + '},\n            '
  'yticklabels={' + ','.join(f'{t:g}' + r'\%' for t in diff_ticks) + '},',
  tex)
assert n_subs == 1, f'panel C colourbar ticks: replaced {n_subs}'

out = tex_path / 'collab-heatmap.tex'
out.write_text(tex)
print(f'wrote {out}')
print(r'preamble needed: \usepackage{pgfplots} \usepackage{pgfplotstable} \pgfplotsset{compat=1.18} '
      r'\usepgfplotslibrary{colormaps,colorbrewer,groupplots} \usepackage{opencolor}')
print(f'top-left cell for the caption: observed {collab_observed.iloc[0, 0] / num_edges * 100:.2f}%, '
      f'shuffled {collab_null.iloc[0, 0] / num_edges * 100:.2f}%, '
      f'difference {diff_pp[0, 0]:+.2f} pp')

### Fig 7 caption
Builds the caption from the same matrices as the figure, and asserts its two claims: every diagonal cell is
positive, and outbreak scientists have the largest one.

In [ ]:
## Fig 7 caption, numbers interpolated from the matrices themselves.
pct = lambda v: v / num_edges * 100
diff_pp_frame = collab_diff * 100

## the two claims the caption makes about the shape of panel C
assert (np.diag(diff_pp_frame.values) > 0).all(), 'diagonal is not uniformly positive'
assert diff_pp_frame.values.diagonal().argmax() == 0, 'experts no longer hold the largest diagonal'

collab_caption = (
  'Collaboration mixing among the three author subgroups (outbreak scientists, bellwethers, '
  'newcomers). '
  '\\textbf{(A)} Observed proportions of within- and across-subgroup co-authorship edges in the '
  'COVID-19 dataset. An edge is one unordered pair of co-authors on one paper, counted once per '
  f'paper they share, giving {num_edges:,} edges over the {len(work_type_counts):,} COVID-19 papers '
  'in the observation window. '
  '\\textbf{(B)} Expected proportions under a null model that randomly reassigns subgroup labels to '
  'authorship slots, preserving every team size, the total edge count, and each subgroup\'s share of '
  'the authorship slots ('
  + ', '.join(f'{share * 100:.1f}\\% {name}' for name, share in zip(
      ['outbreak scientist', 'bellwether', 'newcomer'], q))
  + '). '
  '\\textbf{(C)} Difference between (A) and (B) in percentage points. For example, edges between two '
  f'outbreak scientists account for {pct(collab_observed.iloc[0, 0]):.2f}\\% of the observed dataset '
  f'(A) but only {pct(collab_null.iloc[0, 0]):.2f}\\% of the randomized baseline (B), yielding a '
  f'difference of {diff_pp_frame.iloc[0, 0]:.2f} percentage points (top-left cell of panel C). '
  'The positive diagonal in panel (C) indicates assortative mixing within each author subgroup, with '
  'the strongest within-group concentration among outbreak scientists; the largest deficit is between '
  f'bellwethers and newcomers, {abs(diff_pp_frame.loc["bellwether", "newcomer"]):.2f} percentage points '
  'below the baseline.'
)

print(textwrap.fill(collab_caption, 100))

## Revision analyses (Nov 2026)
Queries added during revision, one subsection each. They were run from a shell with the duckdb CLI and
`pyfixest`, so they have no outputs here; install `duckdb` and `pyfixest` to run them in the notebook. They read
`parq_path`, `data_path` and `csv_dir` and write intermediate files to `../analysis/`. The standalone scripts in
`analysis/` cover the later revision analyses.

In [ ]:
import duckdb
REV = repo_path.parent / 'revisions-nov2026' / 'analysis'
REV.mkdir(parents=True, exist_ok=True)
con = duckdb.connect()
con.execute("SET memory_limit='40GB'; SET threads=16;")
q = lambda sql: con.sql(sql).df()
P, D = str(parq_path), str(data_path)
COVID_CONCEPTS = '3008058167, 3006700255, 3007834351'
WORK_TYPES = "'article','journal-article','proceedings-article','posted-content','book-chapter','preprint','review','conference-paper'"

### Item 5: authorship slots vs authors (Fig 7 methods)
Outbreak scientists hold 16.6% of slots but are 7.7% of unique OW authors: 4.14 COVID papers each vs 1.99 / 1.32.

In [ ]:
q(f'''
WITH t AS (SELECT * FROM '{D}/covid_works_authors_ow.parquet')
SELECT author_type, count(*) slots, round(100.0*count(*)/sum(count(*)) over(),1) slot_pct,
       count(DISTINCT author_id) authors, round(100.0*count(DISTINCT author_id)/sum(count(DISTINCT author_id)) over(),1) author_pct,
       round(count(*)/count(DISTINCT author_id),2) papers_per_author
FROM t GROUP BY 1 ORDER BY 1''')

### Item 6: authors and works by window and topic (Appendix C)
Also confirms the expert set: the TW pool (COVID or outbreak-science authors, 388,126) intersected with OW COVID
authors is exactly the 102,794 outbreak scientists; 100,991 of them carry one of the four outbreak concepts (Fig 6).

In [ ]:
con.execute(f'''
CREATE OR REPLACE TEMP TABLE w AS
  SELECT work_id, any_value(y) y, bool_or(c) c, bool_or(o) o FROM (
    SELECT work_id, publication_year y, true c, false o FROM '{D}/covid_works.parquet'
    UNION ALL SELECT work_id, publication_year, false, true FROM '{D}/outbreak_science_works.parquet') GROUP BY 1;
CREATE OR REPLACE TEMP TABLE a AS
  SELECT DISTINCT x.author_id, w.y, w.c, w.o, x.work_id FROM '{D}/works_authors_2015_2022.parquet' x JOIN w USING(work_id);''')
author_counts = q('''
SELECT coalesce(cast(y AS varchar), CASE WHEN win='TW' THEN 'TW 2015-19' ELSE 'OW 2020-22' END) period,
       count(DISTINCT work_id) FILTER (WHERE c) covid_works, count(DISTINCT author_id) FILTER (WHERE c) covid_authors,
       count(DISTINCT work_id) FILTER (WHERE o) os_works, count(DISTINCT author_id) FILTER (WHERE o) os_authors,
       count(DISTINCT work_id) combined_works, count(DISTINCT author_id) combined_authors
FROM (SELECT *, CASE WHEN y < 2020 THEN 'TW' ELSE 'OW' END win FROM a)
GROUP BY GROUPING SETS ((win, y), (win)) ORDER BY win DESC, y NULLS LAST''')
display(author_counts)
display(q(f'''
WITH tw AS (SELECT DISTINCT author_id FROM a WHERE y < 2020),
     ex AS (SELECT DISTINCT author_id FROM '{D}/covid_works_authors_ow.parquet' WHERE is_expert),
     cw AS (SELECT DISTINCT work_id FROM '{D}/works_concepts_2015_2022.parquet'
            WHERE publication_year < 2020 AND concept_id IN (524204448, 107130276, 116675565, 89623803)),
     oa AS (SELECT DISTINCT author_id FROM '{D}/works_authors_2015_2022.parquet' x SEMI JOIN cw USING(work_id) WHERE publication_year < 2020)
SELECT (SELECT count(*) FROM tw) tw_pool, (SELECT count(*) FROM ex) experts,
       (SELECT count(*) FROM ex SEMI JOIN tw USING(author_id)) pool_in_covid_ow,
       (SELECT count(*) FROM ex SEMI JOIN oa USING(author_id)) experts_with_os_concept'''))

### Item 7: sample-construction flow chart (Appendix D, `figures/figS1-exclusion-flow.tex`)
From the raw snapshot: COVID-tagged works 2020-22 -> 453,367. Writes `flow_f.parquet` (one row per tagged work, one flag
per filter), reused by items 8 and 9.

In [ ]:
con.execute(f'''
CREATE OR REPLACE TEMP TABLE cid AS SELECT DISTINCT work_id FROM read_parquet('{P}/works_concepts/*.parquet')
  WHERE concept_id IN ({COVID_CONCEPTS}) AND score > 0.30 AND publication_year BETWEEN 2020 AND 2022;
CREATE OR REPLACE TEMP TABLE fw AS SELECT w.work_id, any_value(w.type) wtype, bool_or(is_retracted) ret, bool_or(is_paratext) para,
  any_value(num_authors) na, bool_or(doi IS NOT NULL) hasdoi
  FROM read_parquet('{P}/works/*.parquet') w SEMI JOIN cid USING(work_id) WHERE publication_year BETWEEN 2020 AND 2022 GROUP BY 1;
CREATE OR REPLACE TEMP TABLE f AS SELECT *, (wtype NOT IN ({WORK_TYPES}) OR wtype IS NULL) f_type, coalesce(ret,false) f_ret,
  coalesce(para,false) f_para, coalesce(na,0)=0 f_n0, coalesce(na,0)>30 f_big, coalesce(na,0)=1 f_solo, NOT hasdoi f_doi FROM fw;
COPY f TO '{REV}/flow_f.parquet';''')
display(q('''
SELECT 'start' step, count(*) n FROM f
UNION ALL SELECT '1 type',       count(*) FILTER (WHERE f_type) FROM f
UNION ALL SELECT '2 retracted',  count(*) FILTER (WHERE NOT f_type AND f_ret) FROM f
UNION ALL SELECT '3 paratext',   count(*) FILTER (WHERE NOT f_type AND NOT f_ret AND f_para) FROM f
UNION ALL SELECT '4 no authors', count(*) FILTER (WHERE NOT (f_type OR f_ret OR f_para) AND f_n0) FROM f
UNION ALL SELECT '5 >30',        count(*) FILTER (WHERE NOT (f_type OR f_ret OR f_para OR f_n0) AND f_big) FROM f
UNION ALL SELECT '6 solo',       count(*) FILTER (WHERE NOT (f_type OR f_ret OR f_para OR f_n0 OR f_big) AND f_solo) FROM f
UNION ALL SELECT '7 no doi',     count(*) FILTER (WHERE NOT (f_type OR f_ret OR f_para OR f_n0 OR f_big OR f_solo) AND f_doi) FROM f
UNION ALL SELECT 'final',        count(*) FILTER (WHERE NOT (f_type OR f_ret OR f_para OR f_n0 OR f_big OR f_solo OR f_doi)) FROM f'''))
display(q('SELECT sum(f_type::int) type_alone, sum(f_ret::int) ret_alone, sum(f_para::int) para_alone, sum(f_n0::int) n0_alone, '
          'sum(f_big::int) big_alone, sum(f_solo::int) solo_alone, sum(f_doi::int) doi_alone FROM f'))
display(q("SELECT coalesce(wtype, 'NULL') wtype, count(*) n FROM f WHERE f_type GROUP BY 1 ORDER BY 2 DESC"))

### Item 8: works removed by the 30-author cap (Discussion, limitations)
The 4,795 >30-author COVID works passing every other filter: authors classified with the paper's own TW pools, 2-yr
citations counted with the cell-66 citing rules. Writes `big.parquet` / `big_bw.parquet`.

In [ ]:
con.execute(f'''
CREATE OR REPLACE TEMP TABLE big AS SELECT f.work_id, f.na, w.pd FROM '{REV}/flow_f.parquet' f
  JOIN (SELECT work_id, any_value(publication_date) pd FROM read_parquet('{P}/works/*.parquet')
        WHERE publication_year BETWEEN 2020 AND 2022 GROUP BY 1) w USING(work_id)
  WHERE f_big AND NOT (f_type OR f_ret OR f_para OR f_doi);
CREATE OR REPLACE TEMP TABLE ba AS SELECT DISTINCT a.work_id, a.author_id FROM read_parquet('{P}/works_authorships/*.parquet') a
  SEMI JOIN big USING(work_id) WHERE a.assigned_institution;
CREATE OR REPLACE TEMP TABLE twpool AS SELECT DISTINCT x.author_id FROM '{D}/works_authors_2015_2022.parquet' x
  JOIN (SELECT work_id FROM '{D}/covid_works.parquet' WHERE publication_year < 2020
        UNION SELECT work_id FROM '{D}/outbreak_science_works.parquet' WHERE publication_year < 2020) USING(work_id);
CREATE OR REPLACE TEMP TABLE twany AS SELECT DISTINCT author_id FROM '{D}/works_authors_2015_2022.parquet' SEMI JOIN ba USING(author_id)
  WHERE publication_year < 2020;
CREATE OR REPLACE TEMP TABLE bac AS SELECT ba.*, CASE WHEN author_id IN (SELECT author_id FROM twpool) THEN 'expert'
  WHEN author_id IN (SELECT author_id FROM twany) THEN 'bellwether' ELSE 'newcomer' END t FROM ba;
CREATE OR REPLACE TEMP TABLE bw AS SELECT work_id, count(*) n, avg((t='expert')::int) fe FROM bac GROUP BY 1;
COPY big TO '{REV}/big.parquet'; COPY bw TO '{REV}/big_bw.parquet';
CREATE OR REPLACE TEMP TABLE r AS SELECT DISTINCT work_id, referenced_work_id FROM read_parquet('{P}/works_referenced_works/*.parquet')
  WHERE referenced_work_id IN (SELECT work_id FROM big);
CREATE OR REPLACE TEMP TABLE cw AS SELECT work_id, any_value(publication_date) cd FROM read_parquet('{P}/works/*.parquet')
  WHERE work_id IN (SELECT work_id FROM r) AND num_authors BETWEEN 1 AND 30 AND type IN ({WORK_TYPES}) GROUP BY 1;
CREATE OR REPLACE TEMP TABLE c AS SELECT b.work_id, count(cw.work_id) FILTER (WHERE date_diff('day', b.pd, cw.cd) BETWEEN -366 AND 732) c2
  FROM big b LEFT JOIN r ON r.referenced_work_id = b.work_id LEFT JOIN cw ON cw.work_id = r.work_id GROUP BY 1;''')
display(q("SELECT t, count(*) slots, round(100.0*count(*)/sum(count(*)) over(),1) pct FROM bac GROUP BY 1 ORDER BY 1"))
display(q('''SELECT CASE WHEN fe=0 THEN '0%' WHEN fe<=0.5 THEN '1-50%' WHEN fe<1 THEN '51-99%' ELSE '100%' END cls,
             count(*) works, round(avg(c2),1) mean_c2, median(c2) med_c2 FROM c JOIN bw USING(work_id) GROUP BY 1 ORDER BY 1'''))
display(q("SELECT count(*) n, round(avg(c2),1) mean_c2, median(c2) med, round(100*avg((c2=0)::int),1) pct_uncited FROM c"))

### Item 9: Fig 4 exclusions by reason, and preprints (Appendix E)
Reproduces Fig 4's per-class counts exactly (213,968 / 110,244 / 16,718 / 4,377). Writes `fig4_reasons.parquet`.

In [ ]:
con.execute(f'''
CREATE OR REPLACE TEMP TABLE pl AS SELECT work_id, any_value(source_id) source_id, any_value(source_type) source_type
  FROM read_parquet('{P}/works_primary_location/*.parquet') WHERE work_id IN (SELECT work_id FROM '{D}/covid_works_ow.parquet') GROUP BY 1;
CREATE OR REPLACE TEMP TABLE v AS SELECT w.work_id, w.type, w.class_, w.publication_year, w.num_cites_2yrs, pl.source_id, pl.source_type,
  j.impact_factor IS NOT NULL has_if FROM '{D}/covid_works_ow.parquet' w LEFT JOIN pl USING(work_id)
  LEFT JOIN '{D}/covid_journal_sources.parquet' j USING(source_id);
COPY (SELECT work_id, type, class_, source_id, source_type, has_if FROM v) TO '{REV}/fig4_reasons.parquet';''')
display(q('''SELECT class_, count(*) n, count(*) FILTER (WHERE has_if) fig4,
  count(*) FILTER (WHERE NOT has_if AND type='preprint') ex_preprint,
  count(*) FILTER (WHERE NOT has_if AND type<>'preprint' AND (source_id IS NULL OR coalesce(source_type,'')<>'journal')) ex_nonjournal,
  count(*) FILTER (WHERE NOT has_if AND type<>'preprint' AND source_type='journal') ex_journal_no_mc FROM v GROUP BY 1 ORDER BY 1'''))
display(q("SELECT publication_year y, class_, round(100.0*avg((type='preprint')::int),1) pct_preprint FROM v GROUP BY ALL ORDER BY 1, 2"))
display(q('''SELECT class_, count(*) n, round(avg(num_cites_2yrs),1) mean, median(num_cites_2yrs) med,
             round(100*avg((num_cites_2yrs=0)::int),1) uncited FROM v WHERE type='preprint' GROUP BY 1 ORDER BY 1'''))

### Item 10: median and IQR of two-year citations (Appendix F)

In [ ]:
q(f'''SELECT coalesce(cast(publication_year AS varchar), '2020-22') y, class_, count(*) n, round(avg(num_cites_2yrs),2) mean,
      median(num_cites_2yrs) med, quantile_disc(num_cites_2yrs,0.25) q1, quantile_disc(num_cites_2yrs,0.75) q3, max(num_cites_2yrs) mx
      FROM '{D}/covid_works_ow.parquet' GROUP BY GROUPING SETS ((publication_year, class_), (class_)) ORDER BY 1, 2''')

### Item 11: citations per 100,000 eligible COVID works in each paper's window (Appendix G)
Pool = COVID-tagged works meeting the citing rules (type + 1-30 authors), published in [-366, +732] days of the paper;
numerator = the paper's 2-yr citations from that pool. Citer dates use the shuffled date when the citer is in the
sample, as in cell 67. Writes `citerate.parquet`.

In [ ]:
con.execute(f'''
CREATE OR REPLACE TEMP TABLE cc AS SELECT DISTINCT work_id FROM read_parquet('{P}/works_concepts/*.parquet')
  WHERE concept_id IN ({COVID_CONCEPTS}) AND score > 0.30 AND publication_year >= 2019;
CREATE OR REPLACE TEMP TABLE citers AS SELECT w.work_id, coalesce(any_value(s.publication_date), any_value(w.publication_date)) d
  FROM read_parquet('{P}/works/*.parquet') w SEMI JOIN cc USING(work_id)
  LEFT JOIN (SELECT work_id, publication_date FROM '{D}/covid_works_ow.parquet') s USING(work_id)
  WHERE w.num_authors BETWEEN 1 AND 30 AND w.type IN ({WORK_TYPES}) GROUP BY w.work_id;
CREATE OR REPLACE TEMP TABLE daily AS SELECT cast(d AS DATE) dy, count(*) n FROM citers GROUP BY 1;
CREATE OR REPLACE TEMP TABLE sample AS SELECT work_id, cast(publication_date AS DATE) d, publication_year y, class_ FROM '{D}/covid_works_ow.parquet';
CREATE OR REPLACE TEMP TABLE pool AS SELECT s.work_id, sum(dl.n) pool FROM sample s
  JOIN daily dl ON dl.dy BETWEEN s.d - INTERVAL 366 DAY AND s.d + INTERVAL 732 DAY GROUP BY 1;
CREATE OR REPLACE TEMP TABLE rr AS SELECT DISTINCT work_id, referenced_work_id FROM read_parquet('{P}/works_referenced_works/*.parquet')
  WHERE referenced_work_id IN (SELECT work_id FROM sample) AND work_id IN (SELECT work_id FROM citers);
CREATE OR REPLACE TEMP TABLE num AS SELECT s.work_id, count(c.work_id) FILTER (WHERE date_diff('day', s.d, cast(c.d AS DATE)) BETWEEN -366 AND 732) c2cov
  FROM sample s LEFT JOIN rr ON rr.referenced_work_id = s.work_id LEFT JOIN citers c ON c.work_id = rr.work_id GROUP BY 1;
CREATE OR REPLACE TEMP TABLE res AS SELECT s.*, p.pool, n.c2cov, 1000.0*n.c2cov/p.pool rate FROM sample s JOIN pool p USING(work_id) JOIN num n USING(work_id);
COPY res TO '{REV}/citerate.parquet';''')
display(q('''SELECT y, class_, count(*) n, round(avg(pool)) mean_pool, round(100*avg(rate),2) mean_per_100k, round(100*median(rate),2) med_per_100k
             FROM res GROUP BY ALL ORDER BY 1, 2'''))
display(q(f"SELECT round(100.0*sum(c2cov)/(SELECT sum(num_cites_2yrs) FROM '{D}/covid_works_ow.parquet'),1) pct_cites_from_covid FROM res"))

### Item 18: fixed-effects citation models and team-size-stratified Figs 4/5 (Appendix H)
Model table: PPML and OLS on log(1+c), nested FE (team size + month -> + modal country -> + venue), SEs clustered by venue.
Modal country = most frequent affiliation country among a paper's authors (ties alphabetical; 'unknown' if none);
venue = primary-location source ('none' if missing). Writes `fe_model.parquet`, `fe_model_results.csv`, the stratified
CSVs, and regenerates `figS2`/`figS3` in the Overleaf working copy via `make_strat_figs.py`.

In [ ]:
con.execute(f'''
CREATE OR REPLACE TEMP TABLE ac AS SELECT work_id, country_code, count(DISTINCT author_id) na
  FROM read_parquet('{P}/works_authorships/*.parquet')
  WHERE assigned_institution AND country_code IS NOT NULL AND work_id IN (SELECT work_id FROM '{D}/covid_works_ow.parquet') GROUP BY 1, 2;
CREATE OR REPLACE TEMP TABLE mc AS SELECT work_id, first(country_code ORDER BY na DESC, country_code) country,
  count(*) n_countries FROM ac GROUP BY 1;
CREATE OR REPLACE TEMP TABLE m AS SELECT w.work_id, w.num_cites_2yrs cites2, w.class_, 1.0*w.num_experts/w.num_authors frac_exp,
  w.num_authors, strftime(w.publication_date, '%Y-%m') ym, w.publication_year y, w.type wtype,
  coalesce(cast(f.source_id AS varchar), 'none') venue, f.has_if, j.impact_factor, coalesce(mc.country, 'unknown') country,
  coalesce(mc.n_countries, 0) n_countries
  FROM '{D}/covid_works_ow.parquet' w LEFT JOIN '{REV}/fig4_reasons.parquet' f USING(work_id)
  LEFT JOIN '{D}/covid_journal_sources.parquet' j ON j.source_id = f.source_id LEFT JOIN mc USING(work_id);
COPY m TO '{REV}/fe_model.parquet';
CREATE OR REPLACE TEMP TABLE ms AS SELECT *, CASE WHEN num_authors=2 THEN 1 WHEN num_authors<=5 THEN 2 WHEN num_authors<=10 THEN 3 ELSE 4 END bin,
  CASE class_ WHEN '0%' THEN 0 WHEN '1-50%' THEN 50 WHEN '51-99%' THEN 99 ELSE 100 END wt FROM m;
COPY (SELECT bin, wt, count(*) n, quantile_cont(impact_factor,0.25) q1, median(impact_factor) med, quantile_cont(impact_factor,0.75) q3,
      min(impact_factor) mn FROM ms WHERE has_if GROUP BY ALL ORDER BY 1, 2) TO '{REV}/strat_if.csv';
COPY (SELECT bin, y, wt, count(*) n, avg(cites2) mean, 1.96*stddev_samp(cites2)/sqrt(count(*)) ci FROM ms GROUP BY ALL ORDER BY 1, 2, 3)
  TO '{REV}/strat_cites.csv';''')

In [ ]:
## the model scripts are the exact files that produced Appendix Table H1 and the robustness numbers in its caption
import subprocess
for script in ('fe_models.py', 'fe_robust.py'):
    print(subprocess.run(['python', script, str(REV / 'fe_model.parquet')], cwd=REV, capture_output=True, text=True).stdout)
overleaf_figs = REV.parent / 'overleaf' / 'Scientific_Reports' / 'figures_v2'
print(subprocess.run(['python', 'make_strat_figs.py', str(overleaf_figs)], cwd=REV, capture_output=True, text=True).stdout)

### Item 17: outbreak scientists defined without Epidemiology (Appendix I)
Drops Epidemiology (`107130276`) from the expertise concepts; outbreak scientists who no longer qualify become
bellwethers, since all of them published during the TW. This leaves 41,176 of 102,794: 38,737 through
infectious disease, outbreak or pandemic, 1,803 through a COVID-19 paper in the TW only, and 636 epidemiology-only
authors kept by a COVID-19 paper in the TW. Team classes use the number of classified authors as the denominator,
as in the main analysis, and `V='base'` reproduces every published number. The SQL is in `analysis/variantA.sql`
and `analysis/variantA_report.sql`. Only-OS teams become too small to report (467 works in 2022), and the Fig 7
comparison uses the ratio of observed to expected edges, since the difference shrinks with the slot share.

In [ ]:
## per-expert concept tags: epi = published under Epidemiology in TW, other = under ID/Outbreak/Pandemic, cov = COVID work in TW
con.execute(f'''
CREATE OR REPLACE TEMP TABLE ex AS SELECT DISTINCT author_id FROM '{D}/covid_works_authors_ow.parquet' WHERE is_expert;
CREATE OR REPLACE TEMP TABLE tags AS SELECT x.author_id, bool_or(c.concept_id=107130276) epi, bool_or(c.concept_id IN (524204448,116675565,89623803)) other
  FROM '{D}/works_authors_2015_2022.parquet' x
  JOIN (SELECT DISTINCT work_id, concept_id FROM '{D}/works_concepts_2015_2022.parquet'
        WHERE publication_year < 2020 AND concept_id IN (524204448,107130276,116675565,89623803)) c USING(work_id)
  WHERE x.publication_year < 2020 AND x.author_id IN (SELECT author_id FROM ex) GROUP BY 1;
CREATE OR REPLACE TEMP TABLE covtw AS SELECT DISTINCT x.author_id FROM '{D}/works_authors_2015_2022.parquet' x
  JOIN (SELECT work_id FROM '{D}/covid_works.parquet' WHERE publication_year < 2020) USING(work_id) WHERE x.author_id IN (SELECT author_id FROM ex);
CREATE OR REPLACE TEMP TABLE e AS SELECT ex.author_id, coalesce(t.epi,false) epi, coalesce(t.other,false) other,
  ex.author_id IN (SELECT author_id FROM covtw) cov FROM ex LEFT JOIN tags t USING(author_id);
COPY e TO '{REV}/expert_tags.parquet';''')
q('''SELECT count(*) experts_now, count(*) FILTER (WHERE epi AND NOT other) epi_only, count(*) FILTER (WHERE epi AND NOT other AND cov) epi_only_kept_by_covid,
     count(*) FILTER (WHERE other OR cov) variant_A, count(*) FILTER (WHERE (other AND NOT epi) OR cov) variant_B_any_epi_dropped FROM e''')

In [ ]:
## base vs Variant A: Fig 2 shares, Fig 4 medians, Fig 5 citations, Fig 7 mixing. variantA.sql builds `auth` (relabelled
## authorships) and `wc` (per-work counts + class); variantA_report.sql prints the summaries.
import os
cwd = os.getcwd(); os.chdir(REV)     # the SQL reads expert_tags.parquet / fe_model.parquet relative to REV
try:
    for V in ('base', 'A'):
        con.execute(f"SET VARIABLE V='{V}'; SET VARIABLE DATA='{D}';")
        con.execute(open('variantA.sql').read())
        print(f'================ {V}')
        for stmt in open('variantA_report.sql').read().split(';'):
            if 'SELECT' in stmt:
                display(con.sql(stmt).df())
        if V == 'A':
            con.execute("COPY (SELECT work_id, cls, ne_cls FROM wc) TO 'variantA_classes.parquet'")
finally:
    os.chdir(cwd)

In [ ]:
## venue portfolios under both definitions: top-20 journals shared by No-OS and Only-OS, and where JAMA/Lancet/NEJM sit
con.execute(f'''
CREATE OR REPLACE TEMP TABLE j AS SELECT f.work_id, f.class_ base_cls, v.cls a_cls, s.source_name FROM '{REV}/fig4_reasons.parquet' f
  JOIN '{REV}/variantA_classes.parquet' v USING(work_id) JOIN '{D}/covid_journal_sources.parquet' s USING(source_id) WHERE f.has_if;
CREATE OR REPLACE TEMP TABLE t20 AS SELECT * FROM (SELECT *, rank() OVER (PARTITION BY v, cls ORDER BY n DESC) r FROM (
  SELECT 'base' v, base_cls cls, source_name, count(*) n FROM j GROUP BY ALL UNION ALL SELECT 'A', a_cls, source_name, count(*) FROM j GROUP BY ALL)) WHERE r <= 20;''')
q('''SELECT v, (SELECT count(*) FROM t20 a JOIN t20 b USING(source_name) WHERE a.v=x.v AND b.v=x.v AND a.cls='0%' AND b.cls='100%') shared_no_only,
     (SELECT count(*) FROM t20 WHERE v=x.v AND cls='0%' AND source_name IN ('JAMA','The Lancet','New England Journal of Medicine')) big3_in_no_os
     FROM (SELECT DISTINCT v FROM t20) x ORDER BY v DESC''')

In [ ]:
## FE models under Variant A: swap class_ AND frac_exp (frac_exp must be recomputed from the relabelled expert count)
import subprocess
con.execute(f'''COPY (SELECT m.* REPLACE (v.cls AS class_, 1.0*v.ne_cls/m.num_authors AS frac_exp)
                FROM '{REV}/fe_model.parquet' m JOIN '{REV}/variantA_classes.parquet' v USING(work_id)) TO '{REV}/fe_model_variantA.parquet';''')
print(subprocess.run(['python', 'fe_models.py', str(REV / 'fe_model_variantA.parquet')], cwd=REV, capture_output=True, text=True).stdout)